# 01 — Exploratory Analysis: Amazon Electronics Reviews

**Dataset:** 130,038 real Amazon Electronics reviews (5-core subset, sampled 1/13)  
**Source:** http://jmcauley.ucsd.edu/data/amazon/  
**Date Range:** 1999-11-23 to 2014-07-23

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Load real data
df = pd.read_csv('../data/amazon_reviews_electronics_5core.csv')
df['reviewDate'] = pd.to_datetime(df['unixReviewTime'], unit='s')
df['reviewYear'] = df['reviewDate'].dt.year
df['reviewMonth'] = df['reviewDate'].dt.month
df['reviewYearMonth'] = df['reviewDate'].dt.to_period('M').astype(str)
df['reviewLength'] = df['reviewText'].fillna('').str.len()
df['hasHelpfulVotes'] = df['helpful_total'] > 0
df['helpfulnessRatio'] = np.where(df['helpful_total'] > 0, df['helpful_upvotes'] / df['helpful_total'], np.nan)

print(f'Records: {len(df):,}')
print(f'Columns: {list(df.columns)}')
print(f'Date range: {df.reviewDate.min().date()} to {df.reviewDate.max().date()}')
print(f'Unique products: {df.asin.nunique():,}')
print(f'Unique reviewers: {df.reviewerID.nunique():,}')

## 1. Review Volume by Year

In [ ]:
yearly = df.groupby('reviewYear').size().reset_index(name='reviewCount')

fig, ax = plt.subplots()
ax.bar(yearly['reviewYear'], yearly['reviewCount'], color='steelblue', edgecolor='black')
ax.set_xlabel('Year')
ax.set_ylabel('Number of Reviews')
ax.set_title('Amazon Electronics Review Volume by Year')
for _, row in yearly.iterrows():
    ax.text(row['reviewYear'], row['reviewCount'] + 200, f"{row['reviewCount']:,}", ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print(yearly.to_string(index=False))

## 2. Rating Distribution

In [ ]:
rating_dist = df['overall'].value_counts().sort_index()
fig, ax = plt.subplots()
colors = ['#d32f2f', '#f57c00', '#fbc02d', '#689f38', '#388e3c']
ax.bar(rating_dist.index, rating_dist.values, color=colors, edgecolor='black')
ax.set_xlabel('Star Rating')
ax.set_ylabel('Count')
ax.set_title('Rating Distribution (Electronics)')
for x, y in zip(rating_dist.index, rating_dist.values):
    pct = y / len(df) * 100
    ax.text(x, y + 500, f'{y:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()
print(f'Average rating: {df.overall.mean():.2f}')

## 3. Helpfulness Patterns

In [ ]:
# Voted vs. unvoted
voted = df[df['hasHelpfulVotes']]
print(f'Reviews with helpfulness votes: {len(voted):,} ({len(voted)/len(df)*100:.1f}%)')
print(f'Overall helpfulness rate: {voted.helpful_upvotes.sum() / voted.helpful_total.sum() * 100:.1f}%')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of helpfulness ratio
axes[0].hist(voted['helpfulnessRatio'].dropna(), bins=20, color='coral', edgecolor='black')
axes[0].set_xlabel('Helpfulness Ratio (upvotes / total votes)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Helpfulness Ratios')
axes[0].axvline(voted['helpfulnessRatio'].mean(), color='darkred', linestyle='--', label=f'Mean: {voted["helpfulnessRatio"].mean():.2f}')
axes[0].legend()

# Helpfulness by rating
help_by_rating = voted.groupby('overall')['helpfulnessRatio'].mean().reset_index()
axes[1].bar(help_by_rating['overall'], help_by_rating['helpfulnessRatio'], color='teal', edgecolor='black')
axes[1].set_xlabel('Star Rating')
axes[1].set_ylabel('Avg Helpfulness Ratio')
axes[1].set_title('Helpfulness by Rating')
for _, row in help_by_rating.iterrows():
    axes[1].text(row['overall'], row['helpfulnessRatio'] + 0.01, f"{row['helpfulnessRatio']:.2f}", ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 4. Temporal Trends — Review Volume & Average Rating Over Time

In [ ]:
monthly = df.groupby('reviewYearMonth').agg(
    reviewCount=('asin', 'size'),
    avgRating=('overall', 'mean')
).reset_index()
monthly['reviewYearMonth'] = pd.to_datetime(monthly['reviewYearMonth'])

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

axes[0].plot(monthly['reviewYearMonth'], monthly['reviewCount'], color='steelblue', linewidth=1.5)
axes[0].fill_between(monthly['reviewYearMonth'], monthly['reviewCount'], alpha=0.3, color='steelblue')
axes[0].set_ylabel('Reviews per Month')
axes[0].set_title('Monthly Review Volume')

axes[1].plot(monthly['reviewYearMonth'], monthly['avgRating'], color='darkgreen', linewidth=1.5)
axes[1].axhline(df['overall'].mean(), color='red', linestyle='--', label=f'Overall mean: {df["overall"].mean():.2f}')
axes[1].set_ylabel('Average Rating')
axes[1].set_title('Monthly Average Rating Trend')
axes[1].legend()

plt.tight_layout()
plt.show()

# Peak month
peak = monthly.loc[monthly['reviewCount'].idxmax()]
print(f'Peak month: {peak["reviewYearMonth"].strftime("%Y-%m")} with {peak["reviewCount"]:,} reviews')

## 5. Review Length Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['reviewLength'], bins=50, color='mediumpurple', edgecolor='black')
axes[0].set_xlabel('Review Length (characters)')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Review Lengths')
axes[0].set_xlim(0, 2000)

# Box plot by rating
df.boxplot(column='reviewLength', by='overall', ax=axes[1])
axes[1].set_xlabel('Star Rating')
axes[1].set_ylabel('Review Length (characters)')
axes[1].set_title('Review Length by Rating')
axes[1].set_ylim(0, 2000)

plt.suptitle('')
plt.tight_layout()
plt.show()

print(f'Median review length: {df.reviewLength.median():.0f} chars')
print(f'Mean review length: {df.reviewLength.mean():.0f} chars')